# Traffic Simulation — Improved Notebook

This notebook is an improved version of the ChatGPT-generated traffic simulation.  
Changes from the original are marked with `# CHANGE:` comments throughout.

In [ ]:
import osmnx as ox
import networkx as nx
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats  # CHANGE (Task 4): added for correlation analysis

In [ ]:
class RoadNetwork:
    """Loads and manages a road network from OpenStreetMap.

    Handles graph loading, travel time computation, and network visualization.
    """

    def __init__(self, address, dist=1000):
        """Load the road network around an address.

        Args:
            address: Street address to center the network on.
            dist: Radius in meters around the address to include.
        """
        # CHANGE: Load the graph only once (original loaded it 3 separate times,
        # wasting API calls and causing inconsistency — only one copy had travel_time set).
        self.graph = ox.graph_from_address(address, dist=dist, network_type='drive')

        # CHANGE: Extract the largest strongly connected component so that every
        # node can reach every other node. The original crashed with NetworkXNoPath
        # because one-way streets make some node pairs unreachable in a directed graph.
        largest_scc = max(nx.strongly_connected_components(self.graph), key=len)
        self.graph = self.graph.subgraph(largest_scc).copy()

        self._add_travel_times()

    def _add_travel_times(self):
        """Add a travel_time attribute (in minutes) to every edge.

        Uses the edge length and speed limit (defaulting to 30 km/h).
        """
        for u, v, k, data in self.graph.edges(data=True, keys=True):
            speed = data.get("maxspeed", 30)
            if isinstance(speed, list):
                speed = speed[0]
            # length is in meters, speed in km/h → convert speed to m/min
            data["travel_time"] = data["length"] / (float(speed) * 1000 / 60)

    def nodes(self):
        """Return a list of all node IDs in the network."""
        return list(self.graph.nodes())

    def plot(self):
        """Plot the road network."""
        fig, ax = ox.plot_graph(self.graph, figsize=(10, 10))
        return fig, ax

In [ ]:
class Car:
    """A single car navigating the road network.

    Each car has a start node, a destination, and a planned path.
    """

    def __init__(self, start, destination):
        self.current_location = start
        self.destination = destination
        self.path = []

    @property
    def has_arrived(self):
        """Check if the car has reached its destination."""
        return self.current_location == self.destination

In [ ]:
class TrafficSimulation:
    """Runs a traffic simulation on a road network.

    Cars are placed at random nodes and navigate to random destinations
    using shortest paths. Congestion is modeled by checking how many cars
    share the same edge — if above a threshold, cars are delayed.
    """

    def __init__(self, network, num_cars=100, congestion_threshold=5):
        """Initialize the simulation.

        Args:
            network: A RoadNetwork instance.
            num_cars: Number of cars to simulate.
            congestion_threshold: Number of cars on an edge before a jam occurs.
        """
        self.network = network
        self.congestion_threshold = congestion_threshold
        nodes = network.nodes()
        self.cars = [Car(random.choice(nodes), random.choice(nodes)) for _ in range(num_cars)]

        # CHANGE (Task 3): Track congestion history across all simulation steps.
        # The original only had a snapshot at the end, making it impossible to see
        # where congestion accumulated over time.
        self.cumulative_edge_counts = {(u, v, k): 0 for u, v, k in network.graph.edges(keys=True)}
        self.history = []  # list of per-step edge count dicts

    def _move_cars(self):
        """Advance each car one step along its path.

        If a car has no path yet, compute the shortest path to its destination.
        If the edge ahead is congested (more cars than the threshold), the car waits.
        """
        for car in self.cars:
            if car.has_arrived:
                continue

            if not car.path:
                # CHANGE: Original included the current node in the path, so pop(0) removed
                # the current location instead of advancing. Now we slice off the first element
                # immediately so that path[0] is always the *next* node to visit.
                car.path = nx.shortest_path(
                    self.network.graph, car.current_location, car.destination, weight="travel_time"
                )[1:]  # [1:] removes current location from path

            if not car.path:
                # Edge case: start == destination after path computation
                continue

            next_node = car.path[0]

            # Count how many other cars are trying to move onto the same edge right now
            cars_on_edge = sum(
                1 for c in self.cars
                if c.current_location == car.current_location
                and c.path
                and c.path[0] == next_node
            )

            if cars_on_edge > self.congestion_threshold:
                # Congested — car waits this step
                continue
            else:
                # CHANGE: Original did pop(0) to get next_node and then moved, but because of
                # Bug 2 the pop happened before the congestion check, so a car that was blocked
                # would re-insert the node it just popped. Now we only pop after confirming movement.
                car.path.pop(0)
                car.current_location = next_node

    def run(self, num_steps=10):
        """Run the simulation for a given number of steps.

        Args:
            num_steps: Number of time steps to simulate.

        Returns:
            self, for method chaining (e.g., sim.run().plot_traffic()).
        """
        # CHANGE: Original didn't return anything — the cars list was inaccessible
        # from outside the function, causing a NameError in the visualization cell.
        for step in range(num_steps):
            self._move_cars()

            # CHANGE (Task 3): Record edge counts at every step for cumulative analysis
            step_counts = self.get_edge_counts()
            self.history.append(step_counts)
            for edge, count in step_counts.items():
                self.cumulative_edge_counts[edge] += count

            # Count cars at each node for status reporting
            active_cars = sum(1 for c in self.cars if not c.has_arrived)
            print(f"Step {step + 1}: {active_cars} cars still in transit")

        return self

    def get_edge_counts(self):
        """Count how many cars are currently on each edge.

        Returns:
            Dictionary mapping (u, v, key) tuples to car counts.
        """
        counts = {(u, v, k): 0 for u, v, k in self.network.graph.edges(keys=True)}
        for car in self.cars:
            if car.path:
                next_node = car.path[0]
                edge_key = (car.current_location, next_node, 0)
                if edge_key in counts:
                    counts[edge_key] += 1
        return counts

    def plot_traffic(self):
        """Visualize the traffic density on the road network.

        Edges are colored and thickened based on how many cars are on them.
        """
        counts = self.get_edge_counts()
        # CHANGE: Original used max(counts.values()) directly, which causes a
        # division-by-zero error if all counts are 0 (e.g., all cars have arrived).
        max_count = max(max(counts.values()), 1)

        edge_colors = []
        edge_widths = []

        # CHANGE: Original iterated over edges but looked up counts using data['osmid']
        # instead of the actual node IDs (u, v). This meant every lookup returned 0
        # and the heatmap was blank. Now we use the correct (u, v, k) tuple.
        for u, v, k, data in self.network.graph.edges(data=True, keys=True):
            count = counts.get((u, v, k), 0)
            edge_colors.append(plt.cm.Reds(count / max_count))
            edge_widths.append(1 + count)

        fig, ax = ox.plot_graph(
            self.network.graph,
            figsize=(10, 10),
            edge_color=edge_colors,
            edge_linewidth=edge_widths,
        )
        return fig, ax

    # ---- CHANGE (Task 3): New visualization methods below ----

    def plot_congestion_heatmap(self):
        """Heatmap of cumulative congestion across all simulation steps.

        Unlike plot_traffic() which shows a single-step snapshot, this shows
        the total number of car-steps each edge experienced over the entire
        simulation. Includes a colorbar legend and uses a light background
        for readability.
        """
        counts = self.cumulative_edge_counts
        max_count = max(max(counts.values()), 1)

        # Build color and width arrays aligned to the edge order
        edge_colors = []
        edge_widths = []
        color_values = []  # raw normalized values for the colorbar

        for u, v, k in self.network.graph.edges(keys=True):
            count = counts.get((u, v, k), 0)
            normalized = count / max_count
            color_values.append(normalized)
            edge_colors.append(plt.cm.YlOrRd(normalized))
            edge_widths.append(1 + 3 * normalized)

        fig, ax = ox.plot_graph(
            self.network.graph,
            figsize=(12, 12),
            bgcolor="white",
            node_color="gray",
            node_size=5,
            edge_color=edge_colors,
            edge_linewidth=edge_widths,
            show=False,
            close=False,
        )

        # Add a colorbar legend
        sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd, norm=plt.Normalize(0, max_count))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
        cbar.set_label("Cumulative car-steps on edge", fontsize=12)
        ax.set_title("Cumulative Traffic Congestion Heatmap", fontsize=14, color="black")
        plt.show()
        return fig, ax

    def get_street_name(self, u, v, k=0):
        """Look up the street name for an edge, falling back to 'Unnamed'."""
        data = self.network.graph.edges[u, v, k]
        name = data.get("name", "Unnamed")
        if isinstance(name, list):
            name = name[0]
        return name

    def plot_top_congested_roads(self, top_n=15):
        """Bar chart of the most congested roads by cumulative car-steps.

        Groups edges by street name and sums their cumulative counts, so
        multi-segment streets are reported as a single entry.

        Args:
            top_n: Number of top roads to display.
        """
        # Aggregate cumulative counts by street name
        street_counts = {}
        for (u, v, k), count in self.cumulative_edge_counts.items():
            name = self.get_street_name(u, v, k)
            street_counts[name] = street_counts.get(name, 0) + count

        # Sort and take top N
        sorted_streets = sorted(street_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
        names, values = zip(*sorted_streets) if sorted_streets else ([], [])

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(range(len(names)), values, color=plt.cm.YlOrRd(
            [v / max(values) for v in values]
        ))
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names)
        ax.invert_yaxis()
        ax.set_xlabel("Cumulative car-steps")
        ax.set_title(f"Top {top_n} Most Congested Roads")
        plt.tight_layout()
        plt.show()
        return fig, ax

    def plot_congestion_over_time(self, top_n=5):
        """Line plot showing how congestion evolves across simulation steps.

        Plots total congestion (all edges) and per-road congestion for the
        top N most congested streets.

        Args:
            top_n: Number of top roads to highlight individually.
        """
        if not self.history:
            print("No history recorded. Run the simulation first.")
            return

        steps = range(1, len(self.history) + 1)

        # Total congestion per step
        total_per_step = [sum(h.values()) for h in self.history]

        # Identify top streets by cumulative count
        street_counts = {}
        for (u, v, k), count in self.cumulative_edge_counts.items():
            name = self.get_street_name(u, v, k)
            street_counts[name] = street_counts.get(name, 0) + count
        top_streets = [name for name, _ in sorted(street_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]]

        # Build edge-to-street mapping
        edge_to_street = {}
        for u, v, k in self.network.graph.edges(keys=True):
            edge_to_street[(u, v, k)] = self.get_street_name(u, v, k)

        # Per-street congestion over time
        street_per_step = {name: [] for name in top_streets}
        for step_counts in self.history:
            street_step_totals = {name: 0 for name in top_streets}
            for edge, count in step_counts.items():
                name = edge_to_street.get(edge, "Unnamed")
                if name in street_step_totals:
                    street_step_totals[name] += count
            for name in top_streets:
                street_per_step[name].append(street_step_totals[name])

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Left: total congestion
        ax1.plot(steps, total_per_step, "k-o", linewidth=2)
        ax1.set_xlabel("Simulation Step")
        ax1.set_ylabel("Total cars on edges")
        ax1.set_title("Total Network Congestion Over Time")
        ax1.grid(True, alpha=0.3)

        # Right: per-street congestion
        for name in top_streets:
            ax2.plot(steps, street_per_step[name], "-o", label=name, linewidth=1.5)
        ax2.set_xlabel("Simulation Step")
        ax2.set_ylabel("Cars on street edges")
        ax2.set_title(f"Top {top_n} Roads — Congestion Over Time")
        ax2.legend(fontsize=8, loc="best")
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        return fig, (ax1, ax2)

## Load and visualize the road network

In [ ]:
# CHANGE: Network is loaded once here and reused everywhere.
# Original loaded it 3 times in separate cells, wasting API calls.
network = RoadNetwork('Avenida Corrientes 1234, Buenos Aires, Argentina', dist=1000)
print(f"Network has {len(network.nodes())} nodes and {network.graph.number_of_edges()} edges")

In [ ]:
network.plot()

## Run the traffic simulation

In [ ]:
sim = TrafficSimulation(network, num_cars=100, congestion_threshold=5)
sim.run(num_steps=10)

## Visualize traffic density

In [ ]:
sim.plot_traffic()

## Improved Visualizations (Task 3)

The visualizations below address the shortcomings of ChatGPT's original plot:
- **Cumulative heatmap**: shows where congestion built up over the *entire* simulation (not just a final snapshot), with a colorbar so colors can be interpreted quantitatively.
- **Top congested roads**: a bar chart that names the most congested streets, making bottlenecks immediately identifiable.
- **Congestion over time**: shows whether congestion is building, steady, or dissipating — and which specific roads are driving it.

In [ ]:
# Cumulative congestion heatmap — light background, colorbar legend
# Interpretation: darker red edges carried more traffic over the full simulation.
# The colorbar maps edge color to the total number of car-steps.
sim.plot_congestion_heatmap()

In [ ]:
# Top congested roads — bar chart by street name
# Interpretation: taller bars = more total car-steps across all segments of that street.
# Multi-segment streets are aggregated so the full road is represented as one entry.
sim.plot_top_congested_roads(top_n=15)

In [ ]:
# Congestion over time — total network + per-road breakdown
# Interpretation:
#   Left panel:  rising line = congestion building; falling = cars arriving at destinations.
#   Right panel: shows which specific streets are the main contributors to congestion
#                at each step. A street that stays high is a persistent bottleneck.
sim.plot_congestion_over_time(top_n=5)

## Task 4a: Empirical Congestion Analysis

Run the simulation many times and aggregate results to find which roads are
**consistently** the most congested — not just in one unlucky run, but across
many random car placements.

In [ ]:
# CHANGE (Task 4a): Run the simulation multiple times and aggregate congestion data.
# A single run is noisy because car placements are random. By averaging across
# many runs, we identify roads that are *structurally* prone to congestion.

NUM_RUNS = 50
NUM_CARS = 100
NUM_STEPS = 15

# Accumulate total car-steps per edge across all runs
aggregate_counts = {(u, v, k): 0 for u, v, k in network.graph.edges(keys=True)}

for run in range(NUM_RUNS):
    sim = TrafficSimulation(network, num_cars=NUM_CARS, congestion_threshold=5)
    sim.run(num_steps=NUM_STEPS)
    for edge, count in sim.cumulative_edge_counts.items():
        aggregate_counts[edge] += count

# Average per run
avg_counts = {edge: total / NUM_RUNS for edge, total in aggregate_counts.items()}

print(f"Completed {NUM_RUNS} simulation runs.")
print(f"Total edges: {len(avg_counts)}")
print(f"Edges with nonzero average congestion: {sum(1 for v in avg_counts.values() if v > 0)}")

In [ ]:
# Visualize the average congestion across all runs as a heatmap
max_avg = max(max(avg_counts.values()), 1)

edge_colors = []
edge_widths = []
for u, v, k in network.graph.edges(keys=True):
    count = avg_counts.get((u, v, k), 0)
    normalized = count / max_avg
    edge_colors.append(plt.cm.YlOrRd(normalized))
    edge_widths.append(1 + 3 * normalized)

fig, ax = ox.plot_graph(
    network.graph,
    figsize=(12, 12),
    bgcolor="white",
    node_color="gray",
    node_size=5,
    edge_color=edge_colors,
    edge_linewidth=edge_widths,
    show=False,
    close=False,
)
sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd, norm=plt.Normalize(0, max_avg))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label("Avg cumulative car-steps per run", fontsize=12)
ax.set_title(f"Average Congestion Heatmap ({NUM_RUNS} runs)", fontsize=14, color="black")
plt.show()

In [ ]:
# Top 20 most congested roads (averaged across all runs), grouped by street name
street_avg = {}
for (u, v, k), count in avg_counts.items():
    data = network.graph.edges[u, v, k]
    name = data.get("name", "Unnamed")
    if isinstance(name, list):
        name = name[0]
    street_avg[name] = street_avg.get(name, 0) + count

sorted_streets = sorted(street_avg.items(), key=lambda x: x[1], reverse=True)[:20]
names, values = zip(*sorted_streets)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(names)), values, color=plt.cm.YlOrRd([v / max(values) for v in values]))
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel("Avg cumulative car-steps per run")
ax.set_title(f"Top 20 Most Congested Roads (averaged over {NUM_RUNS} runs)")
plt.tight_layout()
plt.show()

## Task 4b: Network Metrics as Predictors of Congestion

We compute several network metrics and test which one best correlates with
the empirical congestion data from Task 4a. The candidate metrics (from Sayama Ch. 17):

- **Edge betweenness centrality**: fraction of all shortest paths that pass through an edge. Hypothesis: edges on many shortest paths carry more traffic.
- **Node betweenness centrality** (averaged over endpoints): nodes that bridge different parts of the network funnel traffic through their edges.
- **Degree centrality** (averaged over endpoints): well-connected intersections may attract more traffic.
- **Closeness centrality** (averaged over endpoints): nodes that are "close" to everything may see more through-traffic.

In [ ]:
# CHANGE (Task 4b): Compute network metrics and correlate with empirical congestion.

G = network.graph

# 1. Edge betweenness centrality (weighted by travel_time to match how cars route)
edge_betweenness = nx.edge_betweenness_centrality(G, weight="travel_time")

# 2. Node-level metrics
node_betweenness = nx.betweenness_centrality(G, weight="travel_time")
degree_centrality = nx.degree_centrality(G)
closeness_centrality = nx.closeness_centrality(G, distance="travel_time")

# Build arrays aligned to edges for correlation
edges = list(G.edges(keys=True))

empirical = np.array([avg_counts.get(e, 0) for e in edges])

# Edge betweenness — nx returns (u,v) keys without the multi-edge key,
# so we map (u,v,k) -> (u,v)
ebc = np.array([edge_betweenness.get((u, v), 0) for u, v, k in edges])

# For node metrics, average the two endpoints of each edge
nbc = np.array([(node_betweenness[u] + node_betweenness[v]) / 2 for u, v, k in edges])
dc = np.array([(degree_centrality[u] + degree_centrality[v]) / 2 for u, v, k in edges])
cc = np.array([(closeness_centrality[u] + closeness_centrality[v]) / 2 for u, v, k in edges])

# Compute Spearman rank correlation (robust to nonlinear relationships)
metrics = {
    "Edge Betweenness Centrality": ebc,
    "Node Betweenness Centrality (avg)": nbc,
    "Degree Centrality (avg)": dc,
    "Closeness Centrality (avg)": cc,
}

print("Correlation between network metrics and empirical congestion:")
print("=" * 65)
results = {}
for name, values in metrics.items():
    spearman_r, spearman_p = stats.spearmanr(values, empirical)
    pearson_r, pearson_p = stats.pearsonr(values, empirical)
    results[name] = {"spearman_r": spearman_r, "pearson_r": pearson_r}
    print(f"\n{name}:")
    print(f"  Spearman r = {spearman_r:.4f}  (p = {spearman_p:.2e})")
    print(f"  Pearson  r = {pearson_r:.4f}  (p = {pearson_p:.2e})")

best_metric = max(results.items(), key=lambda x: abs(x[1]["spearman_r"]))
print(f"\n>> Best predictor: {best_metric[0]} (Spearman r = {best_metric[1]['spearman_r']:.4f})")

In [ ]:
# Scatter plots: each metric vs empirical congestion
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, values) in zip(axes.flat, metrics.items()):
    ax.scatter(values, empirical, alpha=0.3, s=10, color="steelblue")
    ax.set_xlabel(name, fontsize=10)
    ax.set_ylabel("Avg congestion (car-steps)", fontsize=10)
    r = results[name]["spearman_r"]
    ax.set_title(f"{name}\nSpearman r = {r:.3f}", fontsize=11)
    ax.grid(True, alpha=0.3)

    # Add trend line
    if np.std(values) > 0:
        z = np.polyfit(values, empirical, 1)
        p = np.poly1d(z)
        x_line = np.linspace(values.min(), values.max(), 100)
        ax.plot(x_line, p(x_line), "r-", linewidth=1.5, alpha=0.7)

plt.suptitle("Network Metrics vs. Empirical Congestion", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the best predictor metric on the road network for comparison
# Side-by-side: empirical congestion (left) vs best metric prediction (right)

best_name = best_metric[0]
best_values = metrics[best_name]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))

# Left: empirical congestion
max_emp = max(empirical.max(), 1)
emp_colors = [plt.cm.YlOrRd(avg_counts.get(e, 0) / max_emp) for e in edges]
emp_widths = [1 + 3 * (avg_counts.get(e, 0) / max_emp) for e in edges]
ox.plot_graph(
    G, ax=ax1, figsize=(10, 10), bgcolor="white", node_color="gray", node_size=5,
    edge_color=emp_colors, edge_linewidth=emp_widths, show=False, close=False,
)
ax1.set_title("Empirical Congestion\n(simulation average)", fontsize=13, color="black")

# Right: best metric
max_bv = max(best_values.max(), 1e-10)
met_colors = [plt.cm.YlOrRd(v / max_bv) for v in best_values]
met_widths = [1 + 3 * (v / max_bv) for v in best_values]
ox.plot_graph(
    G, ax=ax2, figsize=(10, 10), bgcolor="white", node_color="gray", node_size=5,
    edge_color=met_colors, edge_linewidth=met_widths, show=False, close=False,
)
ax2.set_title(f"Predicted Congestion\n({best_name})", fontsize=13, color="black")

plt.tight_layout()
plt.show()

## Task 4c: Validation on a Different Location

Test whether the best-performing network metric from Task 4b also predicts
congestion in a **different area** of Buenos Aires. We load a new network,
run simulations to get empirical congestion, compute the metric, and compare.

In [ ]:
# CHANGE (Task 4c): Load a different Buenos Aires neighborhood for validation.
# Using Palermo — a very different road structure from the Corrientes/downtown area.
network2 = RoadNetwork('Plaza Italia, Buenos Aires, Argentina', dist=1000)
print(f"Validation network: {len(network2.nodes())} nodes, {network2.graph.number_of_edges()} edges")
network2.plot()

In [ ]:
# Run simulations on the validation network
aggregate_counts2 = {(u, v, k): 0 for u, v, k in network2.graph.edges(keys=True)}

for run in range(NUM_RUNS):
    sim2 = TrafficSimulation(network2, num_cars=NUM_CARS, congestion_threshold=5)
    sim2.run(num_steps=NUM_STEPS)
    for edge, count in sim2.cumulative_edge_counts.items():
        aggregate_counts2[edge] += count

avg_counts2 = {edge: total / NUM_RUNS for edge, total in aggregate_counts2.items()}
print(f"Completed {NUM_RUNS} simulation runs on validation network.")

In [ ]:
# Compute the best metric on the validation network and correlate with empirical data
G2 = network2.graph
edges2 = list(G2.edges(keys=True))
empirical2 = np.array([avg_counts2.get(e, 0) for e in edges2])

# Compute all four metrics on the new network
edge_betweenness2 = nx.edge_betweenness_centrality(G2, weight="travel_time")
node_betweenness2 = nx.betweenness_centrality(G2, weight="travel_time")
degree_centrality2 = nx.degree_centrality(G2)
closeness_centrality2 = nx.closeness_centrality(G2, distance="travel_time")

metrics2 = {
    "Edge Betweenness Centrality": np.array([edge_betweenness2.get((u, v), 0) for u, v, k in edges2]),
    "Node Betweenness Centrality (avg)": np.array([(node_betweenness2[u] + node_betweenness2[v]) / 2 for u, v, k in edges2]),
    "Degree Centrality (avg)": np.array([(degree_centrality2[u] + degree_centrality2[v]) / 2 for u, v, k in edges2]),
    "Closeness Centrality (avg)": np.array([(closeness_centrality2[u] + closeness_centrality2[v]) / 2 for u, v, k in edges2]),
}

print(f"Validation results — does {best_name} still predict congestion?")
print("=" * 65)
for name, values in metrics2.items():
    r, p = stats.spearmanr(values, empirical2)
    marker = " << BEST from original" if name == best_name else ""
    print(f"  {name}: Spearman r = {r:.4f}  (p = {p:.2e}){marker}")

In [ ]:
# Side-by-side comparison on the validation network
best_values2 = metrics2[best_name]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))

# Left: empirical congestion on validation network
max_emp2 = max(empirical2.max(), 1)
emp_colors2 = [plt.cm.YlOrRd(avg_counts2.get(e, 0) / max_emp2) for e in edges2]
emp_widths2 = [1 + 3 * (avg_counts2.get(e, 0) / max_emp2) for e in edges2]
ox.plot_graph(
    G2, ax=ax1, figsize=(10, 10), bgcolor="white", node_color="gray", node_size=5,
    edge_color=emp_colors2, edge_linewidth=emp_widths2, show=False, close=False,
)
ax1.set_title("Empirical Congestion — Plaza Italia area\n(simulation average)", fontsize=13, color="black")

# Right: best metric prediction
max_bv2 = max(best_values2.max(), 1e-10)
met_colors2 = [plt.cm.YlOrRd(v / max_bv2) for v in best_values2]
met_widths2 = [1 + 3 * (v / max_bv2) for v in best_values2]
ox.plot_graph(
    G2, ax=ax2, figsize=(10, 10), bgcolor="white", node_color="gray", node_size=5,
    edge_color=met_colors2, edge_linewidth=met_widths2, show=False, close=False,
)
ax2.set_title(f"Predicted Congestion — Plaza Italia area\n({best_name})", fontsize=13, color="black")

plt.tight_layout()
plt.show()

In [ ]:
# Top congested roads in the validation area
street_avg2 = {}
for (u, v, k), count in avg_counts2.items():
    data = G2.edges[u, v, k]
    name = data.get("name", "Unnamed")
    if isinstance(name, list):
        name = name[0]
    street_avg2[name] = street_avg2.get(name, 0) + count

sorted_streets2 = sorted(street_avg2.items(), key=lambda x: x[1], reverse=True)[:15]
names2, values2 = zip(*sorted_streets2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(names2)), values2, color=plt.cm.YlOrRd([v / max(values2) for v in values2]))
ax.set_yticks(range(len(names2)))
ax.set_yticklabels(names2)
ax.invert_yaxis()
ax.set_xlabel("Avg cumulative car-steps per run")
ax.set_title(f"Top 15 Most Congested Roads — Plaza Italia area ({NUM_RUNS} runs)")
plt.tight_layout()
plt.show()